# Day 02: Production Data Quality, Quarantine & Validation Framework
> **Part 2 of 10 in the PySpark Production Data Engineering Series**

[![PySpark](https://img.shields.io/badge/PySpark-3.5%2B%20%2F%204.0%2B-F26522?style=for-the-badge&logo=apachespark&logoColor=white)](https://spark.apache.org/)
[![Python](https://img.shields.io/badge/Python-3.10%2B-3776AB?style=for-the-badge&logo=python&logoColor=white)](https://python.org)
[![Status](https://img.shields.io/badge/Module-Day%2002%20of%2010-008080?style=for-the-badge)](../README.md)

---

## 🛡️ Day 02: Production Data Quality, Quarantine & Validation Framework
### 🎯 Learning Objectives:
1. Design a resilient **Production Data Quality (DQ) Engine** that isolates corrupt records without crashing the ETL pipeline.
2. Formulate explicit validation rules:
   - Mandatory foreign keys (`customer_id is NOT NULL`)
   - Domain bounds (`quantity > 0`, `price > 0`)
   - Timestamp integrity (ISO-8601 parsing & validation)
   - Uniqueness constraints (`order_id` deduplication)
3. Implement a **Quarantine / Dead-Letter Queue (DLQ)** pattern:
   - Tagging failure reasons in a new `rejection_reason` column.
   - Writing pristine records to `silver_orders_day2`.
   - Diverting tainted records to `rejected_orders_day2` for audit.
4. Generate an automated **Data Quality Reconciliation Report** and configure standard Python `logging`.
---


### ⚙️ Initialize SparkSession for Day 02


In [39]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName("Day2_DataQualityPipeline")
    .master("local[*]")
    .getOrCreate()
)

print("Spark started")

Spark started


In [40]:
import pandas as pd

In [41]:
data = {
    "order_id": [
        2001,
        2002,
        2003,
        2004,
        2005,
        2006,
        2006,
        2007,
        2008,
        2009
    ],

    "customer_id": [
        101,
        102,
        103,
        None,
        105,
        106,
        106,
        107,
        108,
        109
    ],

    "customer_name": [
        "Rahul",
        "Amit",
        "Sneha",
        "Priya",
        "Arjun",
        "Riya",
        "Riya",
        "Karan",
        "Neha",
        "Sourav"
    ],

    "city": [
        "Kolkata",
        "Delhi",
        "Mumbai",
        "Pune",
        "Bangalore",
        "Chennai",
        "Chennai",
        None,
        "Hyderabad",
        "Kolkata"
    ],

    "product": [
        "Laptop",
        "Mouse",
        "Keyboard",
        "Monitor",
        "Laptop",
        "Mouse",
        "Mouse",
        "Keyboard",
        "Laptop",
        "Monitor"
    ],

    "category": [
        "Electronics",
        "Accessories",
        "Accessories",
        "Electronics",
        "Electronics",
        "Accessories",
        "Accessories",
        "Accessories",
        "Electronics",
        "Electronics"
    ],

    "quantity": [
        1,
        2,
        -1,
        1,
        None,
        3,
        3,
        0,
        2,
        1
    ],

    "price": [
        55000,
        800,
        1500,
        12000,
        55000,
        800,
        800,
        -1500,
        55000,
        None
    ],

    "order_date": [
        "2026-09-20",
        "2026-09-20",
        "2026-09-21",
        "2026-09-21",
        "2026-09-22",
        "2026-09-23",
        "2026-09-23",
        "2026-09-24",
        "INVALID_DATE",
        "2026-09-25"
    ]
}

df = pd.DataFrame(data)

df.to_csv(
    "orders_day2.csv",
    index=False
)

df

,order_id,customer_id,customer_name,city,product,category,quantity,price,order_date
0,2001,101.0,Rahul,Kolkata,Laptop,Electronics,1.0,55000.0,2026-09-20
1,2002,102.0,Amit,Delhi,Mouse,Accessories,2.0,800.0,2026-09-20
2,2003,103.0,Sneha,Mumbai,Keyboard,Accessories,-1.0,1500.0,2026-09-21
3,2004,NaN,Priya,Pune,Monitor,Electronics,1.0,12000.0,2026-09-21
4,2005,105.0,Arjun,Bangalore,Laptop,Electronics,NaN,55000.0,2026-09-22
5,2006,106.0,Riya,Chennai,Mouse,Accessories,3.0,800.0,2026-09-23
6,2006,106.0,Riya,Chennai,Mouse,Accessories,3.0,800.0,2026-09-23
7,2007,107.0,Karan,None,Keyboard,Accessories,0.0,-1500.0,2026-09-24
8,2008,108.0,Neha,Hyderabad,Laptop,Electronics,2.0,55000.0,INVALID_DATE
9,2009,109.0,Sourav,Kolkata,Monitor,Electronics,1.0,NaN,2026-09-25


In [42]:
df

,order_id,customer_id,customer_name,city,product,category,quantity,price,order_date
0,2001,101.0,Rahul,Kolkata,Laptop,Electronics,1.0,55000.0,2026-09-20
1,2002,102.0,Amit,Delhi,Mouse,Accessories,2.0,800.0,2026-09-20
2,2003,103.0,Sneha,Mumbai,Keyboard,Accessories,-1.0,1500.0,2026-09-21
3,2004,NaN,Priya,Pune,Monitor,Electronics,1.0,12000.0,2026-09-21
4,2005,105.0,Arjun,Bangalore,Laptop,Electronics,NaN,55000.0,2026-09-22
5,2006,106.0,Riya,Chennai,Mouse,Accessories,3.0,800.0,2026-09-23
6,2006,106.0,Riya,Chennai,Mouse,Accessories,3.0,800.0,2026-09-23
7,2007,107.0,Karan,None,Keyboard,Accessories,0.0,-1500.0,2026-09-24
8,2008,108.0,Neha,Hyderabad,Laptop,Electronics,2.0,55000.0,INVALID_DATE
9,2009,109.0,Sourav,Kolkata,Monitor,Electronics,1.0,NaN,2026-09-25


In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       10 non-null     int64  
 1   customer_id    9 non-null      float64
 2   customer_name  10 non-null     object 
 3   city           9 non-null      object 
 4   product        10 non-null     object 
 5   category       10 non-null     object 
 6   quantity       9 non-null      float64
 7   price          9 non-null      float64
 8   order_date     10 non-null     object 
dtypes: float64(3), int64(1), object(5)
memory usage: 848.0+ bytes


In [44]:
order_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("customer_name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("product", StringType(), True),
    StructField("category", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("order_date", StringType(), True)
])

In [45]:
StructField("order_id", IntegerType(), False)

StructField('order_id', IntegerType(), False)

In [46]:
raw_df=(
    spark.read
    .option("header",True)
    .schema(order_schema)
    .csv("orders_day2.csv")
)

In [47]:
raw_df.show(truncate=False)

+--------+-----------+-------------+---------+--------+-----------+--------+-------+------------+
|order_id|customer_id|customer_name|city     |product |category   |quantity|price  |order_date  |
+--------+-----------+-------------+---------+--------+-----------+--------+-------+------------+
|2001    |NULL       |Rahul        |Kolkata  |Laptop  |Electronics|NULL    |55000.0|2026-09-20  |
|2002    |NULL       |Amit         |Delhi    |Mouse   |Accessories|NULL    |800.0  |2026-09-20  |
|2003    |NULL       |Sneha        |Mumbai   |Keyboard|Accessories|NULL    |1500.0 |2026-09-21  |
|2004    |NULL       |Priya        |Pune     |Monitor |Electronics|NULL    |12000.0|2026-09-21  |
|2005    |NULL       |Arjun        |Bangalore|Laptop  |Electronics|NULL    |55000.0|2026-09-22  |
|2006    |NULL       |Riya         |Chennai  |Mouse   |Accessories|NULL    |800.0  |2026-09-23  |
|2006    |NULL       |Riya         |Chennai  |Mouse   |Accessories|NULL    |800.0  |2026-09-23  |
|2007    |NULL      

In [48]:
total_records=raw_df.count()

In [49]:
print("Total records:",total_records)

Total records: 10


In [50]:
invalid_customer=raw_df.filter(
    col("customer_id").isNull()
)
invalid_customer.show()

+--------+-----------+-------------+---------+--------+-----------+--------+-------+------------+
|order_id|customer_id|customer_name|     city| product|   category|quantity|  price|  order_date|
+--------+-----------+-------------+---------+--------+-----------+--------+-------+------------+
|    2001|       NULL|        Rahul|  Kolkata|  Laptop|Electronics|    NULL|55000.0|  2026-09-20|
|    2002|       NULL|         Amit|    Delhi|   Mouse|Accessories|    NULL|  800.0|  2026-09-20|
|    2003|       NULL|        Sneha|   Mumbai|Keyboard|Accessories|    NULL| 1500.0|  2026-09-21|
|    2004|       NULL|        Priya|     Pune| Monitor|Electronics|    NULL|12000.0|  2026-09-21|
|    2005|       NULL|        Arjun|Bangalore|  Laptop|Electronics|    NULL|55000.0|  2026-09-22|
|    2006|       NULL|         Riya|  Chennai|   Mouse|Accessories|    NULL|  800.0|  2026-09-23|
|    2006|       NULL|         Riya|  Chennai|   Mouse|Accessories|    NULL|  800.0|  2026-09-23|
|    2007|       NUL

In [51]:
valid_customer=raw_df.filter(
    col("customer_id").isNotNull()
)
valid_customer.show()

+--------+-----------+-------------+----+-------+--------+--------+-----+----------+
|order_id|customer_id|customer_name|city|product|category|quantity|price|order_date|
+--------+-----------+-------------+----+-------+--------+--------+-----+----------+
+--------+-----------+-------------+----+-------+--------+--------+-----+----------+



In [52]:
invalid_quantity=raw_df.filter(
    col("quantity").isNull() |
    (col("quantity")<=0)
)
invalid_quantity.select(
    "order_id",
    "quantity"
).show()

+--------+--------+
|order_id|quantity|
+--------+--------+
|    2001|    NULL|
|    2002|    NULL|
|    2003|    NULL|
|    2004|    NULL|
|    2005|    NULL|
|    2006|    NULL|
|    2006|    NULL|
|    2007|    NULL|
|    2008|    NULL|
|    2009|    NULL|
+--------+--------+



In [53]:
invalid_price=raw_df.filter(
    col("price").isNull() |
    (col("price")<=0)
)
invalid_price.select(
    "order_id",
    "price"
).show()

+--------+-------+
|order_id|  price|
+--------+-------+
|    2007|-1500.0|
|    2009|   NULL|
+--------+-------+



In [54]:
validated_df = raw_df.withColumn(
    "parsed_date",
    try_to_date(col("order_date"), "yyyy-MM-dd")
)
display(validated_df)

DataFrame[order_id: int, customer_id: int, customer_name: string, city: string, product: string, category: string, quantity: int, price: double, order_date: string, parsed_date: date]

In [55]:
validated_df.select(
    "order_id",
    "order_date",
    "parsed_date"
).show()

+--------+------------+-----------+
|order_id|  order_date|parsed_date|
+--------+------------+-----------+
|    2001|  2026-09-20| 2026-09-20|
|    2002|  2026-09-20| 2026-09-20|
|    2003|  2026-09-21| 2026-09-21|
|    2004|  2026-09-21| 2026-09-21|
|    2005|  2026-09-22| 2026-09-22|
|    2006|  2026-09-23| 2026-09-23|
|    2006|  2026-09-23| 2026-09-23|
|    2007|  2026-09-24| 2026-09-24|
|    2008|INVALID_DATE|       NULL|
|    2009|  2026-09-25| 2026-09-25|
+--------+------------+-----------+



In [56]:
invalid_date = validated_df.filter(
    col("parsed_date").isNull()
)

invalid_date.select(
    "order_id",
    "order_date"
).show()

+--------+------------+
|order_id|  order_date|
+--------+------------+
|    2008|INVALID_DATE|
+--------+------------+



In [57]:
duplicate_orders = (
    raw_df
    .groupBy("order_id")
    .count()
    .filter(col("count") > 1)
)

duplicate_orders.show()

+--------+-----+
|order_id|count|
+--------+-----+
|    2006|    2|
+--------+-----+



In [58]:
validation_condition=(
    col("customer_id").isNull()
    |
    col("quantity").isNull()
    |
    (col("quantity")<=0)
    |
    col("price").isNull()
    |
    (col("price")<=0)
    |
    col("parsed_date").isNull()
)

In [59]:
invalid_df=validated_df.filter(
    validation_condition
)

In [60]:
valid_df=validated_df.filter(
    ~validation_condition
)

In [61]:
print("Total:", validated_df.count())
print("Valid:", valid_df.count())
print("Invalid:", invalid_df.count())

Total: 10
Valid: 0
Invalid: 10


In [62]:
rejected_df = (
    validated_df
    .withColumn(
        "rejection_reason",
        when(
            col("customer_id").isNull(),
            lit("customer_id is NULL")
        )
        .when(
            col("quantity").isNull(),
            lit("quantity is NULL")
        )
        .when(
            col("quantity") <= 0,
            lit("quantity must be greater than 0")
        )
        .when(
            col("price").isNull(),
            lit("price is NULL")
        )
        .when(
            col("price") <= 0,
            lit("price must be greater than 0")
        )
        .when(
            col("parsed_date").isNull(),
            lit("Invalid order date")
        )
    )
)

In [63]:
rejected_df = rejected_df.filter(
    col("rejection_reason").isNotNull()
)

In [64]:
rejected_df.select(
    "order_id",
    "customer_id",
    "quantity",
    "price",
    "order_date",
    "rejection_reason"
).show(truncate=False)

+--------+-----------+--------+-------+------------+-------------------+
|order_id|customer_id|quantity|price  |order_date  |rejection_reason   |
+--------+-----------+--------+-------+------------+-------------------+
|2001    |NULL       |NULL    |55000.0|2026-09-20  |customer_id is NULL|
|2002    |NULL       |NULL    |800.0  |2026-09-20  |customer_id is NULL|
|2003    |NULL       |NULL    |1500.0 |2026-09-21  |customer_id is NULL|
|2004    |NULL       |NULL    |12000.0|2026-09-21  |customer_id is NULL|
|2005    |NULL       |NULL    |55000.0|2026-09-22  |customer_id is NULL|
|2006    |NULL       |NULL    |800.0  |2026-09-23  |customer_id is NULL|
|2006    |NULL       |NULL    |800.0  |2026-09-23  |customer_id is NULL|
|2007    |NULL       |NULL    |-1500.0|2026-09-24  |customer_id is NULL|
|2008    |NULL       |NULL    |55000.0|INVALID_DATE|customer_id is NULL|
|2009    |NULL       |NULL    |NULL   |2026-09-25  |customer_id is NULL|
+--------+-----------+--------+-------+------------

In [65]:
valid_df=(
    validated_df
    .filter(~validation_condition)
)

In [66]:
rejected_df=(
    rejected_df
    .select(
        "*"
    )
)

In [67]:
valid_df = valid_df.dropDuplicates(
    ["order_id"]
)

In [68]:
silver_df = (
    valid_df
    .withColumn(
        "order_date",
        col("parsed_date")
    )
    .withColumn(
        "revenue",
        col("quantity") * col("price")
    )
    .drop("parsed_date")
)

In [69]:
silver_df.show()

+--------+-----------+-------------+----+-------+--------+--------+-----+----------+-------+
|order_id|customer_id|customer_name|city|product|category|quantity|price|order_date|revenue|
+--------+-----------+-------------+----+-------+--------+--------+-----+----------+-------+
+--------+-----------+-------------+----+-------+--------+--------+-----+----------+-------+



In [70]:
silver_df.write \
    .mode("overwrite") \
    .parquet("silver_orders_day2")

In [71]:
rejected_df.write \
    .mode("overwrite") \
    .parquet("rejected_orders_day2")

In [72]:
total = raw_df.count()
valid = silver_df.count()
rejected = rejected_df.count()

valid_percentage = (valid / total) * 100 if total > 0 else 0.0
rejected_percentage = (rejected / total) * 100 if total > 0 else 0.0

print("========== DATA QUALITY REPORT ==========")
print("Total records    :", total)
print("Valid records    :", valid)
print("Rejected records :", rejected)
print(f"Valid %          : {valid_percentage:.2f}")
print(f"Rejected %       : {rejected_percentage:.2f}")

========== DATA QUALITY REPORT ==========
Total records    : 10
Valid records    : 0
Rejected records : 10
Valid %          : 0.00
Rejected %       : 100.00


In [73]:
rejected_df.groupBy(
    "rejection_reason"
).count().orderBy(
    desc("count")
).show(truncate=False)

+-------------------+-----+
|rejection_reason   |count|
+-------------------+-----+
|customer_id is NULL|10   |
+-------------------+-----+



In [74]:
rejection_rate = rejected / total

if rejection_rate > 0.30:
    pipeline_status = "FAILED"
else:
    pipeline_status = "SUCCESS"

print("Pipeline status:", pipeline_status)

Pipeline status: FAILED


In [75]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger("DataPipeline")

In [76]:
logger.info("Pipeline started")
logger.info(f"Total records received: {total}")
logger.info(f"Valid records: {valid}")
logger.info(f"Rejected records: {rejected}")
logger.info(f"Rejection rate: {rejection_rate:.2%}")
logger.info(f"Pipeline status: {pipeline_status}")

2026-10-08 07:30:47,734 - INFO - Pipeline started
2026-10-08 07:30:47,736 - INFO - Total records received: 10
2026-10-08 07:30:47,739 - INFO - Valid records: 0
2026-10-08 07:30:47,742 - INFO - Rejected records: 10
2026-10-08 07:30:47,745 - INFO - Rejection rate: 100.00%
2026-10-08 07:30:47,748 - INFO - Pipeline status: FAILED


---
### 🎓 Day 02 Complete!
Great job completing this module!
👉 **Next Up:** [**Day 03: Incremental Ingestion, Watermarking & Metadata Tracking**](./Day_03_Incremental_Ingestion_and_Metadata_Audit.ipynb)
